# 01 — Synthetic Transformer Telemetry Generation

This notebook generates 180 days of hourly sensor telemetry for a 25 MVA, 132/33 kV
power transformer, mimicking what an IoT gateway would stream from a real asset
in production.

**Why synthetic data?** Real transformer telemetry is proprietary and not publicly
available. The generator models the physical relationships between load, ambient
temperature, hotspot temperature, dissolved gas concentrations, and degradation
based on textbook physics (IEEE C57.91 thermal model + Arrhenius aging). This
gives us realistic data that the rest of the pipeline can be validated against.

**Sensor channels generated:**
- Electrical: load, voltages, currents
- Thermal: ambient, oil (top/bottom), winding hotspot
- DGA: H2, CH4, C2H2, C2H4, C2H6, CO, CO2
- Other: vibration, partial discharge, OLTC tap position

Two known faults are injected into the timeline so downstream models can be
validated end-to-end.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from src.data_generator import generate_transformer_data
from src.visualization import setup_style, plot_load_and_temperature

setup_style()

## Generate 180 days of telemetry

In [ ]:
df = generate_transformer_data(days=180, seed=42)
df.to_csv('../data/synthetic_transformer_telemetry.csv', index=False)
print(f"Shape: {df.shape}")
print(f"Date range: {df.timestamp.min()} to {df.timestamp.max()}")
df.head()

## Fault label distribution

Most rows are normal operation. Two fault windows are injected:
- A thermal fault <300C around day 90
- A partial discharge event around day 140


In [ ]:
df.fault_label.value_counts().sort_index()

## Visualize the first 30 days: load profile and hotspot temperature

In [ ]:
fig = plot_load_and_temperature(df, hours=24 * 30)
plt.show()

## Verify physical realism

A few sanity checks: the simulated hotspot should respond to load and ambient temperature in expected ways.


In [ ]:
import numpy as np
print("Sensor statistics:")
print(df[["load_pu", "ambient_temp_c", "winding_hotspot_c",
          "oil_temp_top_c", "h2_ppm", "vibration_rms_mm_s"]].describe().round(2))

In [ ]:
# Correlation between load and hotspot - should be strongly positive
corr = df["load_pu"].corr(df["winding_hotspot_c"])
print(f"Correlation(load, hotspot) = {corr:.3f}  (expect > 0.8)")

## Next step

Notebook **02_physics_model** uses this telemetry as input to the physics-based
thermal and aging models, and computes the *residual* between physics prediction
and the simulated measurement. That residual is the key feature the ML layer uses
in notebook 03.
